In [ ]:
import torch
from torch import Tensor, nn
from typing import Dict, Optional
from math import ceil
import numpy as np
from torch.autograd import Function
import os
from torch.autograd.function import Function
from functools import partial
import triton
from copy import deepcopy
import triton.language as tl

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(DEVICE)

In [ ]:
ACT_INTERFACE = {
    "gelu": nn.GELU(),
    "leaky_relu": nn.LeakyReLU(),
    "relu": nn.ReLU(),
    "sigmoid": nn.Sigmoid(),
    "silu": nn.SiLU(),
    "swish": nn.SiLU(),
    "tanh": nn.Tanh(),
    "no_act": nn.Identity(),
}


def eager_forward(
    x: Tensor,
    W_up: Tensor,
    b_up: Optional[Tensor],
    W_gp: Tensor,
    b_gp: Optional[Tensor],
    act_fn: str,
    dropout_p: float,
) -> Tensor:
    
    up = x @ W_up.T
    if b_up is not None:
        up += b_up
        
    gated = x @ W_gp.T
    if b_gp is not None:
        gated += b_gp
        
    gated = ACT_INTERFACE[act_fn]((gated))
    out = gated * up
    # x = nn.functional.dropout(x, dropout_p, training)
    return out


class NaiveGatedMLP(nn.Module):
    def __init__(
        self,
        hidden_size: int = 256,
        intermediate_size: int = 512,
        bias: bool = True,
        hidden_act: str = "silu",
        dropout_p: float = 0.0,
        **kargs,
    ):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=bias)
        self.act = ACT_INTERFACE[hidden_act]
        self.act_fn = hidden_act
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        hidden_states = self.dropout(self.act(self.gate_proj(x)) * self.up_proj(x))
        down_proj = self.down_proj(hidden_states)
        return down_proj


In [ ]:

@triton.jit()
def _compute_sigma(x):
    return 1 / (1 + tl.exp(-x))

@triton.jit()
def _silu_fwd(x):
    return x * _compute_sigma(x)

@triton.jit()
def _act_fwd(x, act_name: tl.constexpr):
    if act_name == "no_act":
        return x
    elif act_name == "silu":
        return _silu_fwd(x)
    else:
        raise NotImplementedError()

@triton.jit()
def map_pid_m_n(pid, M, N, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE_M, optimize_L2):
    if optimize_L2:
        pid_m, pid_n = map_pid_m_n_L2_optim(
            pid, M, N, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE_M
        )
    else:
        n_programs_n = tl.cdiv(N, BLOCK_SIZE_N)
        pid_m = pid // n_programs_n
        pid_n = pid % n_programs_n
    return (pid_m, pid_n)


@triton.jit()
def map_pid_m_n_L2_optim(pid, M, N, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE_M):

    num_blocks_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_blocks_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_programs_in_group = GROUP_SIZE_M * num_blocks_n

    group_size_m = GROUP_SIZE_M
    offset_m = pid // num_programs_in_group
    group_size_m = min(GROUP_SIZE_M, num_blocks_m - offset_m * GROUP_SIZE_M)

    pid_m = ((pid % num_programs_in_group) % group_size_m) + offset_m * GROUP_SIZE_M
    pid_n = (pid % num_programs_in_group) // group_size_m

    return (pid_m, pid_n)


@triton.jit()
def _fwd_kernel(
    x_ptr,
    WT_up_ptr,
    b_up_ptr,
    WT_gp_ptr,
    b_gp_ptr,
    out_ptr,
    act_fn,
    dropout_p,
    M,
    N,
    K,
    NUM_SMS: tl.constexpr,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):

    pid = tl.program_id(axis=0)
    num_programs_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_programs_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_programs_k = tl.cdiv(K, BLOCK_SIZE_K)
    total_programs = num_programs_m * num_programs_n

    ### define tensor descriptors
    has_bias_up = True if b_up_ptr is not None else False
    has_bias_gp = True if b_gp_ptr is not None else False
    
    x_desc = tl.make_tensor_descriptor(
        x_ptr, shape=[M, K], strides=[K, 1], block_shape=[BLOCK_SIZE_M, BLOCK_SIZE_K]
    )

    WT_up_desc = tl.make_tensor_descriptor(
        WT_up_ptr, shape=[N, K], strides=[K, 1], block_shape=[BLOCK_SIZE_N, BLOCK_SIZE_K]
    )

    WT_gp_desc = tl.make_tensor_descriptor(
        WT_gp_ptr, shape=[N, K], strides=[K, 1], block_shape=[BLOCK_SIZE_N, BLOCK_SIZE_K]
    )

    out_desc = tl.make_tensor_descriptor(
        out_ptr, shape=[M, N], strides=[N, 1], block_shape=[BLOCK_SIZE_M, BLOCK_SIZE_N]
    )

    if has_bias_up:
        b_up_desc = tl.make_tensor_descriptor(
            b_up_desc, shape=[N, 1], strides=[1, 0], block_shape=[BLOCK_SIZE_N]
        )

    if has_bias_gp:
        b_gp_desc = tl.make_tensor_descriptor(
            b_gp_desc, shape=[N, 1], strides=[1, 0], block_shape=[BLOCK_SIZE_N]
        )
        
    ### persistent matmul: loop over multiple (m,n) tiles
    for tile_id in tl.range(pid, total_programs, NUM_SMS, flatten=True, warp_specialize=True):
        pid_m, pid_n = map_pid_m_n(
            tile_id, M, N, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE_M, True
        )
        
        offset_m = pid_m * BLOCK_SIZE_M
        offset_n = pid_n * BLOCK_SIZE_N
        
        tile_up = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
        tile_gp = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
        
        for k in tl.range(0, num_programs_k):
            
            offset_k = k * BLOCK_SIZE_K
            
            tile_x = x_desc.load([offset_m, offset_k])
            tile_WT_up = WT_up_desc.load([offset_n, offset_k])
        
            tile_up = tl.dot(tile_x, tile_WT_up.T, acc=tile_up)
            if has_bias_up:
                tile_up += ...
                
            tile_WT_gp = WT_gp_desc.load([offset_n, offset_k])
            tile_gp = tl.dot(tile_x, tile_WT_gp.T, acc=tile_gp)
            if has_bias_gp:
                tile_gp += ...
                
        ### compute act(tile_gp) * tile_up (skip dropout for now)
        tile_out = _act_fwd(tile_gp, act_fn) * tile_up
        
        out_desc.store([offset_m, offset_n], tile_out)
        
                

def validate_dimensions(
    x: Tensor,
    WT_up: Tensor,  # this one is transposed
    b_up: Tensor | None,
    WT_gp: Tensor,  # this one is transposed
    b_gp: Tensor | None,
) -> None:
    assert x.ndims <= 2, f"input tensor must have ndims <=2, got {x.ndims}"
    assert WT_up.shape[1] == x.shape[1], "dimension mismatch in WT_up or x"
    assert WT_gp.shape == WT_up.shape, "dimension mismatch in WT_up or WT_gp"

    if b_up is not None:
        assert b_up.shape[0] == WT_up.shape[0], "dimension mismatch in b_up"

    if b_gp is not None:
        assert b_gp.shape[0] == WT_up.shape[0], "dimension mismatch in b_gp"


def pad_tensor_16_byte_aligned(t: Tensor, axis: int) -> Tensor:
    assert t.ndim == 2, f"expected tensor to have exactly 2 dimensions, got {t.ndims}"
    old_dims = t.shape
    dim = old_dims[axis]
    padded_dim = dim + 16 - dim % 16
    new_dims = (padded_dim, t.shape[1]) if axis == 0 else (t.shape[0], padded_dim)
    new_t = torch.zeros(new_dims, dtype=t.dtype, device=t.device)
    new_t[: old_dims[0], : old_dims[1]] = t
    return new_t


def get_num_streaming_multiprocessors() -> int:
    return (
        10  # dummy value for dev/debugging
        if not torch.cuda.is_available()
        else torch.cuda.get_device_properties("cuda:0").multi_processor_count
    )


def mlp_hidden_states_fwd(
    x: Tensor,
    WT_up: Tensor,  # this one is transposed
    b_up: Tensor | None,
    WT_gp: Tensor,  # this one is transposed
    b_gp: Tensor | None,
    act_fn: str,
    dropout_p: float,
) -> Tensor:
    """
    This function computes the follwing operations in a fused fashion:

        self.dropout(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        
    Wieghts WT_up and WT_gp are kept transposed, and they are transposed back inside the 
    triton kernel when doing tl.dot(x, W.T, ...) 
    """

    ### validate input dimension
    validate_dimensions(x, WT_up, b_up, WT_gp, b_gp)

    M, K = x.shape
    N, _ = WT_up.shape

    ### triton tensor_descriptor needs tensors to have stride(0) that
    # is a multiple of 16. Check this and pad if needed.
    if K % 16 != 0:
        raise NotImplementedError()
        # a = pad_tensor_16_byte_aligned(a, axis=1)
        # b = pad_tensor_16_byte_aligned(b, axis=0)

    if N % 16 != 0:
        raise NotImplementedError()
        # b = pad_tensor_16_byte_aligned(b, axis=1)
        # old_N = N

    ### Create the grid
    NUM_SMS = get_num_streaming_multiprocessors()
    BLOCK_SIZE_M, BLOCK_SIZE_N, BLOCK_SIZE_K, GROUP_SIZE_M = (64, 64, 64, 8)
    grid = (min(NUM_SMS, math.ceil(N / BLOCK_SIZE_M) * math.ceil(N / BLOCK_SIZE_N)),)

    ### custom allocation function
    def allocator(size, stream: int, allignment: Optional[int]):
        return torch.empty(size, device=x.device, dtype=torch.int8)

    triton.set_allocator(allocator)

    ### allocate output and run the kernel
    out = torch.zeros((M, N), dtype=x.dtype, device=x.device)

    _fwd_kernel[grid](
        x,
        WT_up,
        b_up,
        WT_gp,
        b_gp,
        out,
        act_fn,
        dropout_p,
        M,
        N,
        K,
        NUM_SMS,
        BLOCK_SIZE_M,
        BLOCK_SIZE_N,
        BLOCK_SIZE_K,
        GROUP_SIZE_M,
    )

    ###
    return out.to(x.dtype) if x.dtype != out.dtype else out


In [ ]:
MAP_FUN = {
        "torch_eager": eager_forward,
        "torch_module" : None, # must be instantiated in the testing function
        "triton_desc_persistent": mlp_hidden_states_fwd, 
}

matmul_names = list(MAP_FUN.keys())
available_colors = ["black", "red", "green", "blue", "yellow", "cyan", "purple", "grey"]
colors = available_colors[:len(MAP_FUN)]


In [ ]:
def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

ref_lib = 'cuBLAS' if is_cuda() else 'rocBLAS'
TORCH_HAS_FP8 = hasattr(torch, "float8_e5m2")

configs = []
for fp8_inputs in [True]:
    configs.append(
        triton.testing.Benchmark(
            x_names=["M", "N", "K"],  # Argument names to use as an x-axis for the plot
            x_vals=[int(2 ** i) for i in np.arange(5, 15, 1)],  # Different possible values for `x_name`
            line_arg="provider",  # Argument name whose value corresponds to a different line in the plot
            # Possible values for `line_arg`
            # Don't compare to cublas for fp8 cases as torch.matmul doesn't support fp8 at the moment.
            line_vals=matmul_names, #["triton"] if fp8_inputs else [ref_lib.lower(), "triton"],  # Label name for the lines
            line_names=matmul_names, #["Triton"] if fp8_inputs else [ref_lib, "Triton"],  # Line styles
            styles=[(x, "-") for x, _ in zip(colors,matmul_names)], # ("green", "-"), ("blue", "-")],
            ylabel="TFLOPS",  # Label name for the y-axis
            plot_name="matmul-performance-" +
            ("fp16" if not fp8_inputs else "fp8"),  # Name for the plot, used also as a file name for saving the plot.
            args={"fp8_inputs": fp8_inputs},
        ))

@triton.testing.perf_report(configs)
def benchmark(M, N, K, provider, fp8_inputs):
    
    DTYPE = torch.float16
    print(f"{provider=}, {DTYPE=}, {M=}, {N=}, {K=}")
    
    x = torch.randn((M, K), device=DEVICE, dtype=DTYPE)
    # b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
    
    gmlp = NaiveGatedMLP(
        hidden_act='silu', 
        hidden_size=N, 
        intermediate_size=N, 
        bias=False, 
        dropout_p=0.,
        dtype=DTYPE,
        device=DEVICE,
    )#.to(DEVICE).to(DTYPE)
    
    if provider == "triton_module":
        fwd = gmlp()
    else:
        fwd = partial(
            MAP_FUN[provider], 
            W_up=gmlp.up_proj.weight, 
            b_up=gmlp.up_proj.bias, 
            W_gp=gmlp.gate_proj.weight, 
            b_gp=gmlp.gate_proj.bias, 
            act_fn=gmlp.act_fn, 
            dropout_p=gmlp.dropout.p,
        )
    # if TORCH_HAS_FP8 and fp8_inputs:
    #     a = a.to(torch.float8_e5m2)
    #     b = b.T
    #     b = b.to(torch.float8_e5m2)
    quantiles = [0.5, 0.2, 0.8]
    try:
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: fwd(x), quantiles=quantiles) 
    except:
        ms, min_ms, max_ms = 3 * [0]

    perf = lambda ms: 4 * M * N * K * 1e-12 / (ms * 1e-3)
    return perf(ms), perf(max_ms), perf(min_ms)


benchmark.run(show_plots=True, print_data=True)